In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(
    os.getenv("NEO4J_URI"),
    os.getenv("NEO4J_USERNAME"),
    os.getenv("NEO4J_PASSWORD"),
    os.getenv("NEO4J_DATABASE")
)

In [ ]:
from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

graph = Neo4jGraph()

# Text-to-Cypher

- Text-to-Cypher는 자연어 질의를 Neo4j 데이터베이스 쿼리 언어인 Cypher로 변환하여 지식 그래프를 효과적으로 검색할 수 있는 기술을 말한다.


## 자연어를 cypher query로 변환

In [ ]:
################
#  직접 구현
################

graph.refresh_schema()
schema = graph.schema  # str
print(schema)
vector_index = graph.query("SHOW VECTOR INDEXES")
print(vector_index)

In [ ]:
model = ChatOpenAI(model="gpt-5.4-mini")
prompt_template = PromptTemplate(
    template="""
<instruction>
다음 그래프 스키마를 참고해서 질문에 맞는 cypher 쿼리를 작성하세요.
</instruction>

<input_data>
    <schema>{schema}</schema>
    <question>{query}</question>
</input_data>

<constraint>
- 날짜의 경우 `date` 타입으로 변환하여 처리한다.
- 질문과 schema 가 맞지 않으면 vector-index를 참고해서 vector 검색쿼리를 생성한다.
- 생성한 쿼리만 문자열로 반환한다. 설명과 같은 다른 어떠한 것도 추가로 작성하지 않는다.
- 만약 감독의 이름, 영화제목, 배우이름이 쿼리문에 들어갈 경우 영어로 바꿔서 입력한다.
</constraint>
"""
)

query_chain = prompt_template | model | StrOutputParser()

def text_to_query(question: str, schema: str) -> str:
    """자연어 질의를 cypher query로 변환하는 함수"""
    return query_chain.invoke({"schema":schema, "query":question})


In [ ]:
query = "톰 행크스가 출연한 영화는?"
# query = "톰 행크스가 출연한 영화들의 감독들은 누구인가요?"
# query = "2차 세계대전을 배경으로 치열한 전투를 보여주는 영화의 제목과 그 영화 감독, 배우를 알려줘."

cypher_query = text_to_query(query, schema)
print(cypher_query)

print("생성된 쿼리로 조회")
resp = graph.query(cypher_query)
print("-------------조회결과--------------------")
pprint(resp)

# `GraphCypherQAChain` 클래스 사용

- `from langchain_neo4j import GraphCypherQAChain`

## 기능

`GraphCypherQAChain`은 자연어로 Neo4j 데이터베이스와 대화할 수 있도록 하는 체인이다. LLM과 DB 스키마를 활용해 사용자 질문을 Cypher 쿼리로 변환하고 이를 DB에 실행한다.

내부적으로 자연어를 cypher 쿼리로 만드는 것과 질문과 조회결과를 바탕으로 최종 답변을 생성할 것 두 단계로 실행된다.

```
사용자 자연어 질문
    ↓ [1단계 LLM] 그래프 스키마 참고
Cypher 쿼리 생성
    ↓ Neo4j 실행
쿼리 결과 (컨텍스트)
    ↓ [2단계 LLM] 컨텍스트 + 원래 질문
최종 자연어 답변
```


## 객체 생성 — `from_llm()` 클래스 메소드 사용

- 객체 initializer 대신 **`from_llm()` 클래스 메소드**를 사용하여 생성한다.

    ```python
    chain = GraphCypherQAChain.from_llm(
        llm=None,
        *,
        graph,                        # 필수
        allow_dangerous_requests,     # 필수 (True 명시 필요)
        qa_prompt=None,
        cypher_prompt=None,
        cypher_llm=None,
        qa_llm=None,
        exclude_types=[],
        include_types=[],
        validate_cypher=False,
        qa_llm_kwargs=None,
        cypher_llm_kwargs=None,
        use_function_response=False,
        function_response_system=...,
        top_k=10,
        return_intermediate_steps=False,
        return_direct=False,
        verbose=False,
    )
    ```

### 주요 파라미터 정리

**필수 파라미터**

| 파라미터 | 타입 | 설명 |
|---|---|---|
| `graph` | `Neo4jGraph` | 연결할 Neo4j 그래프 객체 |
| `allow_dangerous_requests` | `bool` | `True` 필수. DB 변경 쿼리 허용에 대한 사용자 동의 명시 |

**LLM 설정**

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `llm` | `None` | Cypher 생성 + 답변 생성에 동일한 LLM 사용 |
| `cypher_llm` | `None` | Cypher 생성 전용 LLM (llm과 중복 사용 불가) |
| `qa_llm` | `None` | 답변 생성 전용 LLM (llm과 중복 사용 불가) |

**프롬프트 커스터마이징**

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `cypher_prompt` | (CYPHER_GENERATION_PROMPT) | Cypher 생성용 커스텀 프롬프트 (`{schema}`, `{question}` 변수 포함). 생략시 내장된 프롬프트 사용 |
| `qa_prompt` | (CYPHER_QA_PROMPT) | 답변 생성용 커스텀 프롬프트. 생략시 내장된 프롬프트 사용 |
| `function_response_system` | (FUNCTION_RESPONSE_SYSTEM) | `use_function_response=True` 일 때 시스템 메시지 |

**스키마 필터링**

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `include_types` | `[]` | LLM에 노출할 노드/관계 타입만 지정 |
| `exclude_types` | `[]` | LLM에서 제외할 노드/관계 타입 지정 |

**결과 제어**

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `top_k` | `10` | 쿼리 결과를 LLM에 전달할 최대 건수 |
| `return_intermediate_steps` | `False` | `True`이면 생성된 Cypher, 쿼리 결과도 함께 반환 |
| `return_direct` | `False` | `True`이면 LLM 2단계 생략, DB 조회 결과를 직접 반환 |
| `validate_cypher` | `False` | `True`이면 관계 방향 오류 등을 자동 교정 |
| `use_function_response` | `False` | `True`이면 DB 결과를 tool/function 응답 형식으로 LLM에 전달 (정확도 향상) |
| `verbose` | `False` | `True`이면 생성된 Cypher, 컨텍스트 등 중간 과정 출력 |


## 주요 메소드, 속성

- `invoke()` — 질의응답 실행
  - 기본 반환 값 구조
     ```python
     result = chain.invoke({"query": "Top Gun에 출연한 배우는?"})
     print(result["result"])   # 최종 자연어 답변
     ```

  - `return_intermediate_steps=True` 시 반환 값 구조:

     ```python
     {
        "query": "Top Gun에 출연한 배우는?",
        "result": "Tom Cruise, Val Kilmer ...",
        "intermediate_steps": [
           {"query": "MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) WHERE m.name='Top Gun' RETURN a.name"},
           {"context": [{"a.name": "Tom Cruise"}, {"a.name": "Val Kilmer"}, ...]}
        ]
     }
     ```
### 주요 속성

| 속성 | 설명 |
|---|---|
| `graph_schema` | LLM에 전달되는 현재 그래프 스키마 문자열 |
| `input_key` | 입력 키 (기본값: `"query"`) |
| `output_key` | 출력 키 (기본값: `"result"`) |

- Cypher/QA LLM 분리 + 커스텀 프롬프트 하기

    ```python
    from langchain_core.prompts import PromptTemplate

    CYPHER_TEMPLATE = """스키마: {schema}
    질문을 Cypher로만 변환해줘. 설명 없이 쿼리만 출력.
    질문: {question}"""

    chain = GraphCypherQAChain.from_llm(
        graph=graph,
        cypher_llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),  # Cypher 생성 전용
        qa_llm=ChatOpenAI(model="gpt-4o", temperature=0),           # 답변 생성 전용
        cypher_prompt=PromptTemplate(
            input_variables=["schema", "question"],
            template=CYPHER_TEMPLATE
        ),
        top_k=5,
        return_intermediate_steps=True,
        validate_cypher=True,
        allow_dangerous_requests=True,
    )
    ```

In [ ]:
# GraphCypherQAChain 이 사용하는 프롬프트 조회
from langchain_neo4j.chains.graph_qa.cypher import (
    CYPHER_GENERATION_PROMPT,   # 기본 Cypher 생성 프롬프트: 자연어 -> cypher 쿼리 생성 프롬프트
    CYPHER_QA_PROMPT,           # 기본 QA 프롬프트 : cypher 결과 -> 자연어로 변환시 사용할 프롬프트
)
from langchain_core.prompts import PromptTemplate

print(type(CYPHER_GENERATION_PROMPT), type(CYPHER_QA_PROMPT))
# print(CYPHER_GENERATION_PROMPT.template)
# print("="*50)
# print(CYPHER_QA_PROMPT.template)

################################################
# 사용자 정의 프롬프트 생성 
# - 기존 프롬프트를 수정하는 방식으로 만든다.
################################################
custom_prompt_str = CYPHER_GENERATION_PROMPT.template+\
"""
<constraint>
- When filtering by date, convert the data to the `date` type for processing.
</constraint>
"""

custom_cypher_template = PromptTemplate(template=custom_prompt_str)
print(custom_cypher_template.input_variables)


In [ ]:
#################################
# CypherChain생성
#################################
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain

llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.0)

cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    cypher_prompt=custom_cypher_template # PromptTemplate 
)



In [ ]:
print(cypher_chain.graph_schema)
print("--------------입/출력 key-------------------")
print(cypher_chain.input_key,"/", cypher_chain.output_key)


## 쿼리 실행

In [ ]:
answer = cypher_chain.invoke({"query": "영화 'Apollo 13'에 대한 정보를 알려주세요."})

In [ ]:
print(answer)

In [ ]:
from pprint import pprint
pprint(answer['result'])

In [ ]:
answer = cypher_chain.invoke({"query":"Apollo 13에 출연한 배우이름과 그들이 출연한 다른 영화들을 조회해 주세요."})

In [ ]:
print(answer["result"])

In [ ]:
answer = cypher_chain.invoke({"query": "'Christopher Nolan' 감독의 영화를 모두 찾아주세요."})

In [ ]:
print(answer)

In [ ]:
pprint(answer['result'])

In [ ]:
answer = cypher_chain.invoke({"query": "'Tom Hanks'가 출연한, 평점이 가장 높은 영화 3개는 무엇인가요? 영화제목과 평점을 알려주세요."})

In [ ]:
print(answer)

In [ ]:
pprint(answer['result'])

In [ ]:
answer = cypher_chain.invoke({"query": "2000년 이후 개봉한 'Drama' 장르 영화 중 평점이 높은 순서로 5개를 영화제목, 개봉일자, 평점항목을 알려주세요."})

In [ ]:
print(answer)

In [ ]:
pprint(answer['result'])

### 출력 갯수를 지정 (top k)

In [ ]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    
    top_k=5,                         # 최대 5개의 결과만 반환
)

answer = cypher_chain.invoke({"query": "'Tom Hanks' 배우가 출연한 영화를 모두 알려주세요."})

In [ ]:
# LLM 답변 출력
pprint(answer['result'])

### 생성된 cypher쿼리 포함 반환

- `return_intermediate_steps`=True

In [ ]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    return_intermediate_steps=True  # 생성된 Cypher 쿼리와 중간 결과 확인
)

answer = cypher_chain.invoke({"query": "평점이 8점 이상인 'Action' 장르 영화는 무엇이 있나요?"})

In [ ]:
for k, v in answer.items():
    print(f"{k}: {v}")

### Neo4j DB 반환결과만 얻기

- `return_direct`=True
- cypher 쿼리 실행 결과를 바탕으로 LLM이 응답메세지를 만들어 반환하는데 `return_direct=True` 로 하면 cypher 쿼리 결과를 그대로 dictionary로 반환한다.

In [ ]:
# 직접 Cypher 결과 얻기 (LLM 답변 없이)
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    return_direct=True   # LLM 답변 생성 단계 건너뛰기
)

answer = cypher_chain.invoke({"query": "2000년대(2000-01-01 ~ 2009-12-31) 개봉한 영화를 평점 순으로 정렬해주세요."})

In [ ]:
for k, v in answer.items():
    print(f"{k}: {v}")

In [ ]:
import pandas as pd
pd.DataFrame([item for item in answer['result']])

### cypher 쿼리 생성 모델과 최종 답변 생성 모델을 다르게 적용

In [ ]:
cypher_llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.0)
qa_llm = ChatOpenAI(model="gpt-5.5", temperature=0.0)

cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=cypher_llm,  # Cypher 쿼리 생성용 LLM
    qa_llm=qa_llm,          # 최종 답변 생성용 LLM
    
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
)

answer = cypher_chain.invoke({"query": "배우 'Leonardo DiCaprio'와 감독 'Christopher Nolan'이 함께 작업한 영화가 있나요?"})

In [ ]:
for k, v in answer.items():
    print(f"{k}: {v}")